### **Inferring**
#### **Setup**

In [1]:
import os, openai
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI
from pprint import pprint


_ = load_dotenv(find_dotenv())
openai.api_key = os.environ['OPENAI_API_KEY']

llm_model = 'gpt-3.5-turbo'
chat = ChatOpenAI(model = llm_model)

#### **Product review text**

In [2]:
lamp_review = '''
Needed a nice lamp for my bedroom, and this one had \
additional storage and not too high of a price point. \
Got it fast.  The string to our lamp broke during the \
transit and the company happily sent over a new one. \
Came within a few days as well. It was easy to put \
together.  I had a missing part, so I contacted their \
support and they very quickly got me the missing piece! \
Lumina seems to me to be a great company that cares \
about their customers and products!!
'''

#### **Sentiment (positive/negative)**

In [5]:
prompt = f"""
What is the sentiment of the following product review, 
which is delimited with triple backticks?

Review text: '''{lamp_review}'''
"""

response = chat.invoke(prompt)
pprint(response.content)

('The sentiment of the review is positive. The customer is satisfied with the '
 'product, the customer service, and the company as a whole.')


In [6]:
prompt = f"""
What is the sentiment of the following product review, 
which is delimited with triple backticks?

Give your answer as a single word, either "positive" \
or "negative".

Review text: '''{lamp_review}'''
"""

response = chat.invoke(prompt)
pprint(response.content)

'Positive'


#### **Identify types of emotions**

In [7]:
prompt = f"""
Identify a list of emotions that the writer of the \
following review is expressing. Include no more than \
five items in the list. Format your answer as a list of \
lower-case words separated by commas.

Review text: '''{lamp_review}'''
"""

response = chat.invoke(prompt)
pprint(response.content)

'happy, satisfied, relieved, grateful, impressed'


#### **Identify anger**

In [9]:
prompt = f"""
Is the writer of the following review expressing anger?\
The review is delimited with triple backticks. \
Give your answer as either yes or no.

Review text: '''{lamp_review}'''
"""

response = chat.invoke(prompt)
pprint(response.content)

'No'


#### **Extract product and company name from customer reviews**

In [10]:
prompt = f"""
Identify the following items from the review text: 
- Item purchased by reviewer
- Company that made the item

The review is delimited with triple backticks. \
Format your response as a JSON object with \
"Item" and "Brand" as the keys. 
If the information isn't present, use "unknown" \
as the value.
Make your response as short as possible.
  
Review text: '''{lamp_review}'''
"""

response = chat.invoke(prompt)
pprint(response.content)

'{\n  "Item": "lamp",\n  "Brand": "Lumina"\n}'


#### **Doing multiple tasks at once**

In [11]:
prompt = f"""
Identify the following items from the review text: 
- Sentiment (positive or negative)
- Is the reviewer expressing anger? (true or false)
- Item purchased by reviewer
- Company that made the item

The review is delimited with triple backticks. \
Format your response as a JSON object with \
"Sentiment", "Anger", "Item" and "Brand" as the keys.
If the information isn't present, use "unknown" \
as the value.
Make your response as short as possible.
Format the Anger value as a boolean.

Review text: '''{lamp_review}'''
"""

response = chat.invoke(prompt)
pprint(response.content)

('```json\n'
 '{\n'
 '    "Sentiment": "positive",\n'
 '    "Anger": false,\n'
 '    "Item": "lamp",\n'
 '    "Brand": "Lumina"\n'
 '}\n'
 '```')


#### **Inferring topics**

In [ ]:
story = '''
In a recent survey conducted by the government, 
public sector employees were asked to rate their level 
of satisfaction with the department they work at. 
The results revealed that NASA was the most popular 
department with a satisfaction rating of 95%.

One NASA employee, John Smith, commented on the findings, 
stating, "I'm not surprised that NASA came out on top. 
It's a great place to work with amazing people and 
incredible opportunities. I'm proud to be a part of 
such an innovative organization."

The results were also welcomed by NASA's management team, 
with Director Tom Johnson stating, "We are thrilled to 
hear that our employees are satisfied with their work at NASA. 
We have a talented and dedicated team who work tirelessly 
to achieve our goals, and it's fantastic to see that their 
hard work is paying off."

The survey also revealed that the 
Social Security Administration had the lowest satisfaction 
rating, with only 45% of employees indicating they were 
satisfied with their job. The government has pledged to 
address the concerns raised by employees in the survey and 
work towards improving job satisfaction across all departments.
'''

#### **Infer 5 topics**

In [13]:
prompt = f"""
Determine five topics that are being discussed in the \
following text, which is delimited by triple backticks.

Make each item one or two words long. 

Format your response as a list of items separated by commas.

Text sample: '''{story}'''
"""

response = chat.invoke(prompt)
pprint(response.content)

('1. Survey\n'
 '2. Employee satisfaction\n'
 '3. NASA\n'
 '4. Social Security Administration\n'
 '5. Job satisfaction')


#### **Make a news alert for certain topics**

In [16]:
topic_list = [
    'NASA', 
    'federal government', 
    'local government', 
    'employee comments', 
    'Social Security Administration'
]

prompt = f"""
Determine whether each item in the following list of \
topics is a topic in the text below, which
is delimited with triple backticks.

Give your answer as follows:
item from the list: 0 or 1

List of topics: {", ".join(topic_list)}

Text sample: '''{story}'''
"""

response = chat.invoke(prompt)
pprint(response.content)

('1: NASA\n'
 '0: federal government\n'
 '0: local government\n'
 '1: employee comments\n'
 '1: Social Security Administration')


In [ ]:
topic_dict = {}
for line in response.content.split('\n'):
    if ': ' in line:
        k, v = line.split(': ', 1)
        if k.isdigit():
            topic_dict[v.lower()] = int(k)

if topic_dict.get('nasa') == 1:
    print('ALERT: New NASA story!')

ALERT: New NASA story!
